In [ ]:
import os
import numpy as np
import nibabel as nib
import seaborn as sns
import matplotlib.pyplot as plt

####-------------------------------------
    
import numpy as np

def process_data_and_get_corr_matrix(num_pairs, v1, v2):
    """
    Process data and return the correlation matrices without leave-one-out iteration.
    """
    corr_matrices = []

    for pair_num in range(num_pairs):
        send_stack = v1[pair_num]
        rece_stack = v2[pair_num]

        # Concatenate the two conditions
        pair_matrix = np.concatenate((send_stack, rece_stack), axis=0)

        # Calculate the correlation matrix
        corr_matrix = np.corrcoef(pair_matrix)

        # Fisher's Z Transformation
        fisher_z_matrix = 0.5 * np.log((1 + corr_matrix) / (1 - corr_matrix))

        # Extract the part of the matrix that correlates the two conditions
        corr_matrix_half = fisher_z_matrix[:send_stack.shape[0], send_stack.shape[0]:]  
        corr_matrices.append(corr_matrix_half)

    corr_matrices = np.array(corr_matrices)

    return corr_matrices


def calculate_z_corr_matrix2(corr_matrices_12, corr_matrices_34):
    corr_matrices_34 = np.array(corr_matrices_34)
    corr_matrices_12 = np.array(corr_matrices_12)
    corr_matrix_all = np.vstack((corr_matrices_12, corr_matrices_34))
    return corr_matrix_all
###---------------------------------------------------------------
import h5py
# Define the file paths
file_path1 = 'face10156_imagery_va.h5'
file_path2 = 'face10156_imagery_vb.h5'
file_path3 = 'face10156_speak_va.h5'
file_path4 = 'face10156_speak_vb.h5'

def load_data(file_path):
    """Function to load data from subarray_0 to subarray_199 from a given file path."""
    with h5py.File(file_path, 'r') as file:
        data_list = []
        for i in range(400):  # Load from subarray_0 to subarray_399
            dataset_name = f'subarray_{i}'
            if dataset_name in file:
                data = file[dataset_name][:]
                data_list.append(data)
            else:
                print(f"Dataset '{dataset_name}' not found in the file.")
        return data_list

# Load data for each file
stacked_var_list = load_data(file_path1)
stacked_vbr_list = load_data(file_path2)
stacked_vas_list = load_data(file_path3)
stacked_vbs_list = load_data(file_path4)

import numpy as np

def check_for_nans(data_list, list_name):
    """Function to check for NaN values in a list of datasets."""
    for i, dataset in enumerate(data_list):
        if np.isnan(dataset).any():
            print(f"NaN found in dataset {i} of {list_name}")

# Check each list for NaN values
check_for_nans(stacked_var_list, 'stacked_var_list')
check_for_nans(stacked_vbr_list, 'stacked_vbr_list')
check_for_nans(stacked_vas_list, 'stacked_vas_list')
check_for_nans(stacked_vbs_list, 'stacked_vbs_list')

all_correlations_real1 = []
all_z_correlations_real = []
all_correlations_real2 = []
all_correlations_real3 = []
all_correlations_real4 = []
all_z_correlations_real_sr = []

num_pairsa=23

for brain_area in range(400):
    # 对于每个脑区，提取相应的数据
    vas_data = stacked_vas_list[brain_area]  # shape (23, 24, number_of_voxels)
    vbr_data = stacked_vbr_list[brain_area]  # shape (23, 24, number_of_voxels)
    var_data = stacked_var_list[brain_area]  # shape (23, 24, number_of_voxels)
    vbs_data = stacked_vbs_list[brain_area]  # shape (23, 24, number_of_voxels)

    # 计算 cross_pair_correlation
    real_corr_sarb = process_data_and_get_corr_matrix(num_pairsa,vas_data, vbr_data)
    all_correlations_real1.append(real_corr_sarb)

    # 计算 average_cross_pair_correlation
    real_corr_rbsa = process_data_and_get_corr_matrix(num_pairsa,vbr_data, vas_data)
    all_correlations_real3.append(real_corr_rbsa)

    real_corr_sbra = process_data_and_get_corr_matrix(num_pairsa,vbs_data, var_data)
    all_correlations_real2.append(real_corr_sbra)

    # 计算 average_cross_pair_correlation
    real_corr_rasb  = process_data_and_get_corr_matrix(num_pairsa,var_data,vbs_data)
    all_correlations_real4.append(real_corr_rasb)
    
    z_corr_matrix_real = calculate_z_corr_matrix2(real_corr_sarb, real_corr_sbra)
    all_z_correlations_real.append(z_corr_matrix_real)
    
    # print("finish real")

import h5py

# Define file paths for saving
file_path5 = 'face10156_z_correlations_real_si.h5'
# Save all_z_correlations_real
with h5py.File(file_path5, 'w') as h5f:
    # Assuming all_z_correlations_real is a list of NumPy arrays
    for i, matrix in enumerate(all_z_correlations_real):
        h5f.create_dataset(f'z_corr_matrix_{i}', data=matrix)
        
print("finish output real")

#-------------------------------
import numpy as np

def calculate_cross_pair_correlation(num_pairs, v1, v2, cdsize):
    all_correlations = []

    for send_pair_num in range(num_pairs):
        send_stack = v1[send_pair_num]
        pair_correlations = []

        for rece_pair_num in range(num_pairs):
            if send_pair_num == rece_pair_num:
                continue  # Skip the same pair

            rece_stack = v2[rece_pair_num]

            # Concatenate the two conditions for this pair
            pair_matrix = np.concatenate((send_stack, rece_stack), axis=0)

            # Compute the correlation matrix
            corr_matrix = np.corrcoef(pair_matrix)

            # Fisher's Z Transformation
            fisher_z_matrix = 0.5 * np.log((1 + corr_matrix) / (1 - corr_matrix))

            # Extract the part of the matrix that correlates the two conditions
            corr_matrix_half = fisher_z_matrix[:cdsize, cdsize:]
            pair_correlations.append(corr_matrix_half)

        all_correlations.append(pair_correlations)

    return all_correlations

def average_cross_pair_correlation(num_pairs, v1, v2,cdsize):
    all_correlations = calculate_cross_pair_correlation(num_pairs, v1, v2, cdsize)
    average_correlations = []
    for send_pair_num in range(num_pairs):
        avg_corr = np.mean(all_correlations[send_pair_num], axis=0)
        average_correlations.append(avg_corr)
    return average_correlations

def grandavg_norealpair_correlation(v1_list, v2_list):
    grandavg_correlations = []
    for send_pair_num in range(len(v1_list)):
        avg_corr = (v1_list[send_pair_num] + v2_list[send_pair_num]) / 2
        grandavg_correlations.append(avg_corr)
    return grandavg_correlations



all_correlations_sarb = []  # 存储每个脑区的 cross_pair_correlation
all_correlations_rbsa = []  # 存储每个脑区的 cross_pair_correlation
all_correlations_sbra = []  # 存储每个脑区的 cross_pair_correlation
all_correlations_rasb = []  # 存储每个脑区的 cross_pair_correlation
all_avg_correlations_sarb = []  # 存储每个脑区的 cross_pair_correlation
all_avg_correlations_rbsa = []  # 存储每个脑区的 cross_pair_correlation
all_avg_correlations_sbra = []  # 存储每个脑区的 cross_pair_correlation
all_avg_correlations_rasb = []  # 存储每个脑区的 cross_pair_correlation

num_pairsa=23

for brain_area in range(400):
    # 对于每个脑区，提取相应的数据
    vas_data = stacked_vas_list[brain_area]  # shape (23, 24, number_of_voxels)
    vbr_data = stacked_vbr_list[brain_area]  # shape (23, 24, number_of_voxels)

    # 计算 cross_pair_correlation
    cross_corr_sarb = calculate_cross_pair_correlation(num_pairsa,vas_data, vbr_data, cdsize=24)
    all_correlations_sarb.append(cross_corr_sarb)

    cross_corr_rbsa = calculate_cross_pair_correlation(num_pairsa,vbr_data, vas_data, cdsize=24)
    all_correlations_rbsa.append(cross_corr_rbsa)


for brain_area in range(400):
    # 对于每个脑区，提取相应的数据
    vbs_data = stacked_vbs_list[brain_area]  # shape (23, 24, number_of_voxels)
    var_data = stacked_var_list[brain_area]  # shape (23, 24, number_of_voxels)

    # 计算 cross_pair_correlation
    cross_corr_sbra = calculate_cross_pair_correlation(num_pairsa,vbs_data, var_data, cdsize=24)
    all_correlations_sbra.append(cross_corr_sbra)


    cross_corr_rasb = calculate_cross_pair_correlation(num_pairsa,var_data, vbs_data, cdsize=24)
    all_correlations_rasb.append(cross_corr_rasb)

# # Define file paths for saving
file_path6 = 'face10156_cross_correlations_si.h5'

# Save the data
with h5py.File(file_path6, 'w') as h5f:
    # Assuming all lists are lists of NumPy arrays
    for i, matrix in enumerate(all_correlations_sarb):
        h5f.create_dataset(f'all_correlations_sarb_{i}', data=matrix)

    for i, matrix in enumerate(all_correlations_rbsa):
        h5f.create_dataset(f'all_correlations_rbsa_{i}', data=matrix)

    for i, matrix in enumerate(all_correlations_sbra):
        h5f.create_dataset(f'all_correlations_sbra_{i}', data=matrix)

    for i, matrix in enumerate(all_correlations_rasb):
        h5f.create_dataset(f'all_correlations_rasb_{i}', data=matrix)
        
print("finish output cross")


/tmp/ipykernel_1028004/3443283951.py:28: RuntimeWarning: divide by zero encountered in divide
  fisher_z_matrix = 0.5 * np.log((1 + corr_matrix) / (1 - corr_matrix))
/home/sylsherry/miniconda3/envs/tfgpu/lib/python3.9/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/sylsherry/miniconda3/envs/tfgpu/lib/python3.9/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/tmp/ipykernel_1028004/3443283951.py:28: RuntimeWarning: divide by zero encountered in log
  fisher_z_matrix = 0.5 * np.log((1 + corr_matrix) / (1 - corr_matrix))


finish output real


/tmp/ipykernel_1028004/3443283951.py:156: RuntimeWarning: divide by zero encountered in divide
  fisher_z_matrix = 0.5 * np.log((1 + corr_matrix) / (1 - corr_matrix))
/tmp/ipykernel_1028004/3443283951.py:156: RuntimeWarning: divide by zero encountered in log
  fisher_z_matrix = 0.5 * np.log((1 + corr_matrix) / (1 - corr_matrix))


In [1]:
import os
import numpy as np
import nibabel as nib
import seaborn as sns
import matplotlib.pyplot as plt

####-------------------------------------
    
import numpy as np

def process_data_and_get_corr_matrix(num_pairs, v1, v2):
    """
    Process data and return the correlation matrices without leave-one-out iteration.
    """
    corr_matrices = []

    for pair_num in range(num_pairs):
        send_stack = v1[pair_num]
        rece_stack = v2[pair_num]

        # Concatenate the two conditions
        pair_matrix = np.concatenate((send_stack, rece_stack), axis=0)

        # Calculate the correlation matrix
        corr_matrix = np.corrcoef(pair_matrix)

        # Fisher's Z Transformation
        fisher_z_matrix = 0.5 * np.log((1 + corr_matrix) / (1 - corr_matrix))

        # Extract the part of the matrix that correlates the two conditions
        corr_matrix_half = fisher_z_matrix[:send_stack.shape[0], send_stack.shape[0]:]  
        corr_matrices.append(corr_matrix_half)

    corr_matrices = np.array(corr_matrices)

    return corr_matrices


def calculate_z_corr_matrix2(corr_matrices_12, corr_matrices_34):
    corr_matrices_34 = np.array(corr_matrices_34)
    corr_matrices_12 = np.array(corr_matrices_12)
    corr_matrix_all = np.vstack((corr_matrices_12, corr_matrices_34))
    return corr_matrix_all
###---------------------------------------------------------------
import h5py
# Define the file paths
file_path1 = 'con10156_imagery_va.h5'
file_path2 = 'con10156_imagery_vb.h5'
file_path3 = 'con10156_speak_va.h5'
file_path4 = 'con10156_speak_vb.h5'

def load_data(file_path):
    """Function to load data from subarray_0 to subarray_199 from a given file path."""
    with h5py.File(file_path, 'r') as file:
        data_list = []
        for i in range(400):  # Load from subarray_0 to subarray_399
            dataset_name = f'subarray_{i}'
            if dataset_name in file:
                data = file[dataset_name][:]
                data_list.append(data)
            else:
                print(f"Dataset '{dataset_name}' not found in the file.")
        return data_list

# Load data for each file
stacked_var_list = load_data(file_path1)
stacked_vbr_list = load_data(file_path2)
stacked_vas_list = load_data(file_path3)
stacked_vbs_list = load_data(file_path4)

import numpy as np

def check_for_nans(data_list, list_name):
    """Function to check for NaN values in a list of datasets."""
    for i, dataset in enumerate(data_list):
        if np.isnan(dataset).any():
            print(f"NaN found in dataset {i} of {list_name}")

# Check each list for NaN values
check_for_nans(stacked_var_list, 'stacked_var_list')
check_for_nans(stacked_vbr_list, 'stacked_vbr_list')
check_for_nans(stacked_vas_list, 'stacked_vas_list')
check_for_nans(stacked_vbs_list, 'stacked_vbs_list')

all_correlations_real1 = []
all_z_correlations_real = []
all_correlations_real2 = []
all_correlations_real3 = []
all_correlations_real4 = []
all_z_correlations_real_sr = []

num_pairsa=23

for brain_area in range(400):
    # 对于每个脑区，提取相应的数据
    vas_data = stacked_vas_list[brain_area]  # shape (23, 24, number_of_voxels)
    vbr_data = stacked_vbr_list[brain_area]  # shape (23, 24, number_of_voxels)
    var_data = stacked_var_list[brain_area]  # shape (23, 24, number_of_voxels)
    vbs_data = stacked_vbs_list[brain_area]  # shape (23, 24, number_of_voxels)

    # 计算 cross_pair_correlation
    real_corr_sarb = process_data_and_get_corr_matrix(num_pairsa,vas_data, vbr_data)
    all_correlations_real1.append(real_corr_sarb)

    # 计算 average_cross_pair_correlation
    real_corr_rbsa = process_data_and_get_corr_matrix(num_pairsa,vbr_data, vas_data)
    all_correlations_real3.append(real_corr_rbsa)

    real_corr_sbra = process_data_and_get_corr_matrix(num_pairsa,vbs_data, var_data)
    all_correlations_real2.append(real_corr_sbra)

    # 计算 average_cross_pair_correlation
    real_corr_rasb  = process_data_and_get_corr_matrix(num_pairsa,var_data,vbs_data)
    all_correlations_real4.append(real_corr_rasb)
    
    z_corr_matrix_real = calculate_z_corr_matrix2(real_corr_sarb, real_corr_sbra)
    all_z_correlations_real.append(z_corr_matrix_real)
    
    # print("finish real")

import h5py

# Define file paths for saving
file_path5 = 'con10156_z_correlations_real_si.h5'
# Save all_z_correlations_real
with h5py.File(file_path5, 'w') as h5f:
    # Assuming all_z_correlations_real is a list of NumPy arrays
    for i, matrix in enumerate(all_z_correlations_real):
        h5f.create_dataset(f'z_corr_matrix_{i}', data=matrix)
        
print("finish output real")

#-------------------------------
import numpy as np

def calculate_cross_pair_correlation(num_pairs, v1, v2, cdsize):
    all_correlations = []

    for send_pair_num in range(num_pairs):
        send_stack = v1[send_pair_num]
        pair_correlations = []

        for rece_pair_num in range(num_pairs):
            if send_pair_num == rece_pair_num:
                continue  # Skip the same pair

            rece_stack = v2[rece_pair_num]

            # Concatenate the two conditions for this pair
            pair_matrix = np.concatenate((send_stack, rece_stack), axis=0)

            # Compute the correlation matrix
            corr_matrix = np.corrcoef(pair_matrix)

            # Fisher's Z Transformation
            fisher_z_matrix = 0.5 * np.log((1 + corr_matrix) / (1 - corr_matrix))

            # Extract the part of the matrix that correlates the two conditions
            corr_matrix_half = fisher_z_matrix[:cdsize, cdsize:]
            pair_correlations.append(corr_matrix_half)

        all_correlations.append(pair_correlations)

    return all_correlations

def average_cross_pair_correlation(num_pairs, v1, v2,cdsize):
    all_correlations = calculate_cross_pair_correlation(num_pairs, v1, v2, cdsize)
    average_correlations = []
    for send_pair_num in range(num_pairs):
        avg_corr = np.mean(all_correlations[send_pair_num], axis=0)
        average_correlations.append(avg_corr)
    return average_correlations

def grandavg_norealpair_correlation(v1_list, v2_list):
    grandavg_correlations = []
    for send_pair_num in range(len(v1_list)):
        avg_corr = (v1_list[send_pair_num] + v2_list[send_pair_num]) / 2
        grandavg_correlations.append(avg_corr)
    return grandavg_correlations



all_correlations_sarb = []  # 存储每个脑区的 cross_pair_correlation
all_correlations_rbsa = []  # 存储每个脑区的 cross_pair_correlation
all_correlations_sbra = []  # 存储每个脑区的 cross_pair_correlation
all_correlations_rasb = []  # 存储每个脑区的 cross_pair_correlation
all_avg_correlations_sarb = []  # 存储每个脑区的 cross_pair_correlation
all_avg_correlations_rbsa = []  # 存储每个脑区的 cross_pair_correlation
all_avg_correlations_sbra = []  # 存储每个脑区的 cross_pair_correlation
all_avg_correlations_rasb = []  # 存储每个脑区的 cross_pair_correlation

num_pairsa=23

for brain_area in range(400):
    # 对于每个脑区，提取相应的数据
    vas_data = stacked_vas_list[brain_area]  # shape (23, 24, number_of_voxels)
    vbr_data = stacked_vbr_list[brain_area]  # shape (23, 24, number_of_voxels)

    # 计算 cross_pair_correlation
    cross_corr_sarb = calculate_cross_pair_correlation(num_pairsa,vas_data, vbr_data, cdsize=24)
    all_correlations_sarb.append(cross_corr_sarb)

    cross_corr_rbsa = calculate_cross_pair_correlation(num_pairsa,vbr_data, vas_data, cdsize=24)
    all_correlations_rbsa.append(cross_corr_rbsa)


for brain_area in range(400):
    # 对于每个脑区，提取相应的数据
    vbs_data = stacked_vbs_list[brain_area]  # shape (23, 24, number_of_voxels)
    var_data = stacked_var_list[brain_area]  # shape (23, 24, number_of_voxels)

    # 计算 cross_pair_correlation
    cross_corr_sbra = calculate_cross_pair_correlation(num_pairsa,vbs_data, var_data, cdsize=24)
    all_correlations_sbra.append(cross_corr_sbra)


    cross_corr_rasb = calculate_cross_pair_correlation(num_pairsa,var_data, vbs_data, cdsize=24)
    all_correlations_rasb.append(cross_corr_rasb)

# # Define file paths for saving
file_path6 = 'con10156_cross_correlations_si.h5'

# Save the data
with h5py.File(file_path6, 'w') as h5f:
    # Assuming all lists are lists of NumPy arrays
    for i, matrix in enumerate(all_correlations_sarb):
        h5f.create_dataset(f'all_correlations_sarb_{i}', data=matrix)

    for i, matrix in enumerate(all_correlations_rbsa):
        h5f.create_dataset(f'all_correlations_rbsa_{i}', data=matrix)

    for i, matrix in enumerate(all_correlations_sbra):
        h5f.create_dataset(f'all_correlations_sbra_{i}', data=matrix)

    for i, matrix in enumerate(all_correlations_rasb):
        h5f.create_dataset(f'all_correlations_rasb_{i}', data=matrix)
        
print("finish output cross")


/tmp/ipykernel_708502/2807302341.py:28: RuntimeWarning: divide by zero encountered in divide
  fisher_z_matrix = 0.5 * np.log((1 + corr_matrix) / (1 - corr_matrix))
/tmp/ipykernel_708502/2807302341.py:28: RuntimeWarning: divide by zero encountered in log
  fisher_z_matrix = 0.5 * np.log((1 + corr_matrix) / (1 - corr_matrix))
/home/sylsherry/miniconda3/envs/tfgpu/lib/python3.9/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/sylsherry/miniconda3/envs/tfgpu/lib/python3.9/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


finish output real


/tmp/ipykernel_708502/2807302341.py:156: RuntimeWarning: divide by zero encountered in divide
  fisher_z_matrix = 0.5 * np.log((1 + corr_matrix) / (1 - corr_matrix))
/tmp/ipykernel_708502/2807302341.py:156: RuntimeWarning: divide by zero encountered in log
  fisher_z_matrix = 0.5 * np.log((1 + corr_matrix) / (1 - corr_matrix))


finish output cross


In [4]:
import os
import numpy as np
import nibabel as nib
import seaborn as sns
import matplotlib.pyplot as plt

####-------------------------------------
    
import numpy as np

def process_data_and_get_corr_matrix(num_pairs, v1, v2):
    """
    Process data and return the correlation matrices without leave-one-out iteration.
    """
    corr_matrices = []

    for pair_num in range(num_pairs):
        send_stack = v1[pair_num]
        rece_stack = v2[pair_num]

        # Concatenate the two conditions
        pair_matrix = np.concatenate((send_stack, rece_stack), axis=0)

        # Calculate the correlation matrix
        corr_matrix = np.corrcoef(pair_matrix)

        # Fisher's Z Transformation
        fisher_z_matrix = 0.5 * np.log((1 + corr_matrix) / (1 - corr_matrix))

        # Extract the part of the matrix that correlates the two conditions
        corr_matrix_half = fisher_z_matrix[:send_stack.shape[0], send_stack.shape[0]:]  
        corr_matrices.append(corr_matrix_half)

    corr_matrices = np.array(corr_matrices)

    return corr_matrices


def calculate_z_corr_matrix2(corr_matrices_12, corr_matrices_34):
    corr_matrices_34 = np.array(corr_matrices_34)
    corr_matrices_12 = np.array(corr_matrices_12)
    corr_matrix_all = np.vstack((corr_matrices_12, corr_matrices_34))
    return corr_matrix_all
###---------------------------------------------------------------
import h5py
# Define the file paths
file_path1 = 'con10156_imagery_va.h5'
file_path2 = 'con10156_imagery_vb.h5'
file_path3 = 'con10156_view_va.h5'
file_path4 = 'con10156_view_vb.h5'

def load_data(file_path):
    """Function to load data from subarray_0 to subarray_199 from a given file path."""
    with h5py.File(file_path, 'r') as file:
        data_list = []
        for i in range(400):  # Load from subarray_0 to subarray_399
            dataset_name = f'subarray_{i}'
            if dataset_name in file:
                data = file[dataset_name][:]
                data_list.append(data)
            else:
                print(f"Dataset '{dataset_name}' not found in the file.")
        return data_list

# Load data for each file
stacked_var_list = load_data(file_path1)
stacked_vbr_list = load_data(file_path2)
stacked_vas_list = load_data(file_path3)
stacked_vbs_list = load_data(file_path4)

import numpy as np

def check_for_nans(data_list, list_name):
    """Function to check for NaN values in a list of datasets."""
    for i, dataset in enumerate(data_list):
        if np.isnan(dataset).any():
            print(f"NaN found in dataset {i} of {list_name}")

# Check each list for NaN values
check_for_nans(stacked_var_list, 'stacked_var_list')
check_for_nans(stacked_vbr_list, 'stacked_vbr_list')
check_for_nans(stacked_vas_list, 'stacked_vas_list')
check_for_nans(stacked_vbs_list, 'stacked_vbs_list')

all_correlations_real1 = []
all_z_correlations_real = []
all_correlations_real2 = []
all_correlations_real3 = []
all_correlations_real4 = []
all_z_correlations_real_sr = []

num_pairsa=23

for brain_area in range(400):
    # 对于每个脑区，提取相应的数据
    vas_data = stacked_vas_list[brain_area]  # shape (23, 24, number_of_voxels)
    vbr_data = stacked_vbr_list[brain_area]  # shape (23, 24, number_of_voxels)
    var_data = stacked_var_list[brain_area]  # shape (23, 24, number_of_voxels)
    vbs_data = stacked_vbs_list[brain_area]  # shape (23, 24, number_of_voxels)

    # 计算 cross_pair_correlation
    real_corr_sarb = process_data_and_get_corr_matrix(num_pairsa,vas_data, vbr_data)
    all_correlations_real1.append(real_corr_sarb)

    # 计算 average_cross_pair_correlation
    real_corr_rbsa = process_data_and_get_corr_matrix(num_pairsa,vbr_data, vas_data)
    all_correlations_real3.append(real_corr_rbsa)

    real_corr_sbra = process_data_and_get_corr_matrix(num_pairsa,vbs_data, var_data)
    all_correlations_real2.append(real_corr_sbra)

    # 计算 average_cross_pair_correlation
    real_corr_rasb  = process_data_and_get_corr_matrix(num_pairsa,var_data,vbs_data)
    all_correlations_real4.append(real_corr_rasb)
    
    z_corr_matrix_real = calculate_z_corr_matrix2(real_corr_sarb, real_corr_sbra)
    all_z_correlations_real.append(z_corr_matrix_real)
    
    # print("finish real")

import h5py

# Define file paths for saving
file_path5 = 'con10156_z_correlations_real_vi.h5'
# Save all_z_correlations_real
with h5py.File(file_path5, 'w') as h5f:
    # Assuming all_z_correlations_real is a list of NumPy arrays
    for i, matrix in enumerate(all_z_correlations_real):
        h5f.create_dataset(f'z_corr_matrix_{i}', data=matrix)
        
print("finish output real")

#-------------------------------
import numpy as np

def calculate_cross_pair_correlation(num_pairs, v1, v2, cdsize):
    all_correlations = []

    for send_pair_num in range(num_pairs):
        send_stack = v1[send_pair_num]
        pair_correlations = []

        for rece_pair_num in range(num_pairs):
            if send_pair_num == rece_pair_num:
                continue  # Skip the same pair

            rece_stack = v2[rece_pair_num]

            # Concatenate the two conditions for this pair
            pair_matrix = np.concatenate((send_stack, rece_stack), axis=0)

            # Compute the correlation matrix
            corr_matrix = np.corrcoef(pair_matrix)

            # Fisher's Z Transformation
            fisher_z_matrix = 0.5 * np.log((1 + corr_matrix) / (1 - corr_matrix))

            # Extract the part of the matrix that correlates the two conditions
            corr_matrix_half = fisher_z_matrix[:cdsize, cdsize:]
            pair_correlations.append(corr_matrix_half)

        all_correlations.append(pair_correlations)

    return all_correlations

def average_cross_pair_correlation(num_pairs, v1, v2,cdsize):
    all_correlations = calculate_cross_pair_correlation(num_pairs, v1, v2, cdsize)
    average_correlations = []
    for send_pair_num in range(num_pairs):
        avg_corr = np.mean(all_correlations[send_pair_num], axis=0)
        average_correlations.append(avg_corr)
    return average_correlations

def grandavg_norealpair_correlation(v1_list, v2_list):
    grandavg_correlations = []
    for send_pair_num in range(len(v1_list)):
        avg_corr = (v1_list[send_pair_num] + v2_list[send_pair_num]) / 2
        grandavg_correlations.append(avg_corr)
    return grandavg_correlations



all_correlations_sarb = []  # 存储每个脑区的 cross_pair_correlation
all_correlations_rbsa = []  # 存储每个脑区的 cross_pair_correlation
all_correlations_sbra = []  # 存储每个脑区的 cross_pair_correlation
all_correlations_rasb = []  # 存储每个脑区的 cross_pair_correlation
all_avg_correlations_sarb = []  # 存储每个脑区的 cross_pair_correlation
all_avg_correlations_rbsa = []  # 存储每个脑区的 cross_pair_correlation
all_avg_correlations_sbra = []  # 存储每个脑区的 cross_pair_correlation
all_avg_correlations_rasb = []  # 存储每个脑区的 cross_pair_correlation

num_pairsa=23

for brain_area in range(400):
    # 对于每个脑区，提取相应的数据
    vas_data = stacked_vas_list[brain_area]  # shape (23, 24, number_of_voxels)
    vbr_data = stacked_vbr_list[brain_area]  # shape (23, 24, number_of_voxels)

    # 计算 cross_pair_correlation
    cross_corr_sarb = calculate_cross_pair_correlation(num_pairsa,vas_data, vbr_data, cdsize=24)
    all_correlations_sarb.append(cross_corr_sarb)

    cross_corr_rbsa = calculate_cross_pair_correlation(num_pairsa,vbr_data, vas_data, cdsize=24)
    all_correlations_rbsa.append(cross_corr_rbsa)


for brain_area in range(400):
    # 对于每个脑区，提取相应的数据
    vbs_data = stacked_vbs_list[brain_area]  # shape (23, 24, number_of_voxels)
    var_data = stacked_var_list[brain_area]  # shape (23, 24, number_of_voxels)

    # 计算 cross_pair_correlation
    cross_corr_sbra = calculate_cross_pair_correlation(num_pairsa,vbs_data, var_data, cdsize=24)
    all_correlations_sbra.append(cross_corr_sbra)


    cross_corr_rasb = calculate_cross_pair_correlation(num_pairsa,var_data, vbs_data, cdsize=24)
    all_correlations_rasb.append(cross_corr_rasb)

# # Define file paths for saving
file_path6 = 'con10156_cross_correlations_vi.h5'

# Save the data
with h5py.File(file_path6, 'w') as h5f:
    # Assuming all lists are lists of NumPy arrays
    for i, matrix in enumerate(all_correlations_sarb):
        h5f.create_dataset(f'all_correlations_sarb_{i}', data=matrix)

    for i, matrix in enumerate(all_correlations_rbsa):
        h5f.create_dataset(f'all_correlations_rbsa_{i}', data=matrix)

    for i, matrix in enumerate(all_correlations_sbra):
        h5f.create_dataset(f'all_correlations_sbra_{i}', data=matrix)

    for i, matrix in enumerate(all_correlations_rasb):
        h5f.create_dataset(f'all_correlations_rasb_{i}', data=matrix)
        
print("finish output cross")


/tmp/ipykernel_708502/101723057.py:28: RuntimeWarning: divide by zero encountered in divide
  fisher_z_matrix = 0.5 * np.log((1 + corr_matrix) / (1 - corr_matrix))
/tmp/ipykernel_708502/101723057.py:28: RuntimeWarning: divide by zero encountered in log
  fisher_z_matrix = 0.5 * np.log((1 + corr_matrix) / (1 - corr_matrix))


finish output real


/tmp/ipykernel_708502/101723057.py:156: RuntimeWarning: divide by zero encountered in divide
  fisher_z_matrix = 0.5 * np.log((1 + corr_matrix) / (1 - corr_matrix))
/tmp/ipykernel_708502/101723057.py:156: RuntimeWarning: divide by zero encountered in log
  fisher_z_matrix = 0.5 * np.log((1 + corr_matrix) / (1 - corr_matrix))


finish output cross


In [3]:
df -h


NameError: name 'df' is not defined